# Stage 3A — Analytical Universe, Denominators, and Method Eligibility

This notebook prepares reproducible analytical views from the committed ownership, acquisition-universe, and standardized-observation records. It separates structural and competitive universes, constructs explicit denominators, evaluates longitudinal eligibility, and identifies persistence baselines without calculating portfolio winners, composite scores, or final performance results.


## Environment Setup

Import the standard libraries used for integrity validation, tabular preparation, deterministic output writing, and packaging. Lock all inputs to the verified Stage 2B publication commit.


In [1]:
from __future__ import annotations

import hashlib
import os
import re
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

REPOSITORY_HEAD = "5e64c377be2deea7c5b3568aefe2c12d9e1db2d8"
REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
PREPARATION_REFERENCE_DATE = "2026-08-21"

REQUIRED_FILES = [
    "metadata/ownership_registry.csv",
    "metadata/performance_acquisition_universe.csv",
    "metadata/category_crosswalk.csv",
    "metadata/brand_label_crosswalk.csv",
    "metadata/comparability_rules.csv",
    "metadata/stage2_mapping_validation.csv",
    "metadata/stage2_standardization_validation.csv",
    "data/standardized/performance_observations.csv",
]

EXPECTED_SHA256 = {
    "metadata/ownership_registry.csv": "a30edb56b67f77857dfdf2332f3b19409db65a7d3771f7edb3af316b30b34744",
    "metadata/performance_acquisition_universe.csv": "17a9126dfa46db43be0b50f4c2f30ea4ee23713526372e05697e66250d42dfb9",
    "metadata/category_crosswalk.csv": "180632c43e9cb0e520f25717f3229f5828bb1bd091af76915043c409ea978b6b",
    "metadata/brand_label_crosswalk.csv": "1ca5acd5a02ebe68609b9970cbffec18cb4e275c8ee2083b9341a5fa7b998963",
    "metadata/comparability_rules.csv": "2e7c0659f2d7addbe41d2028fd88452a217c6bd0b49623fca70c96cb6b466a97",
    "metadata/stage2_mapping_validation.csv": "1026be2930b3555808868fe4f2a0d685d91638d6d47843f3f603dab11932cc55",
    "metadata/stage2_standardization_validation.csv": "c1f3ad599ae4e6a4cd8f3c11482c266c27ebb71f7747b67f55ffd4821afee0b7",
    "data/standardized/performance_observations.csv": "63b30fca6260ab12acc1a15ec7d96acd945c3a2d80de1416f3ff6cd9d0a9cc53",
}

PRIMARY_RELATIONSHIPS = {"controlled", "controlled_group_portfolio"}
ELIGIBLE_CATEGORY_MAPPINGS = {"direct", "resolved_with_caveat"}
FOCAL_GROUPS = ["Wings Group", "Indofood", "Mayora", "Unilever Indonesia"]

OUTPUT_ROOT = Path(os.environ.get("FMCG_STAGE3A_OUTPUT_ROOT", "/content/stage3a_outputs"))
ANALYTICAL_ROOT = OUTPUT_ROOT / "data" / "analytical"
METADATA_ROOT = OUTPUT_ROOT / "metadata"
ANALYTICAL_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)


## Committed Input Retrieval

Load the exact required files from the private repository at the locked commit. The Colab Secret named GITHUB_TOKEN is used only in the authorization header and is never printed, embedded in a URL, or written to disk.


In [2]:
# source_marker: colab_private_repository_input_retrieval
configured_root = os.environ.get("FMCG_STAGE3A_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run this notebook in Google Colab or set FMCG_STAGE3A_INPUT_ROOT for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            "The Colab Secret GITHUB_TOKEN is unavailable or access has not been granted to this notebook."
        )

    INPUT_ROOT = Path("/content/fmcg_stage3a_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in REQUIRED_FILES:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        api_url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}/contents/"
            f"{encoded_path}?ref={REPOSITORY_HEAD}"
        )
        request = urllib.request.Request(
            api_url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage3a-colab",
            },
        )
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing_inputs = [
    relative_path
    for relative_path in REQUIRED_FILES
    if not (INPUT_ROOT / relative_path).exists()
]
if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(REQUIRED_FILES)}/{len(REQUIRED_FILES)}")


Input mode: locked_github_commit
Required files found: 8/8


## Input Integrity Validation

Verify the SHA-256 checksum of every required input before loading or deriving any analytical view.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


checksum_rows = []
for relative_path in REQUIRED_FILES:
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    expected_sha256 = EXPECTED_SHA256[relative_path]
    checksum_rows.append(
        {
            "file_path": relative_path,
            "expected_sha256": expected_sha256,
            "actual_sha256": actual_sha256,
            "status": "passed" if actual_sha256 == expected_sha256 else "failed",
        }
    )

checksum_validation = pd.DataFrame(checksum_rows)
if not checksum_validation["status"].eq("passed").all():
    raise ValueError("Input checksum validation failed.")

print(checksum_validation[["file_path", "status"]].to_string(index=False))


                                     file_path status
               metadata/ownership_registry.csv passed
 metadata/performance_acquisition_universe.csv passed
               metadata/category_crosswalk.csv passed
            metadata/brand_label_crosswalk.csv passed
              metadata/comparability_rules.csv passed
        metadata/stage2_mapping_validation.csv passed
metadata/stage2_standardization_validation.csv passed
data/standardized/performance_observations.csv passed


## Registry and Standardized Observation Loading

Load identifiers and categorical fields as strings. Confirm that the committed Stage 2 mapping and standardization gates contain no critical failures.


In [4]:
def read_table(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(
        INPUT_ROOT / relative_path,
        dtype=str,
        keep_default_na=False,
    )


ownership = read_table("metadata/ownership_registry.csv")
acquisition_universe = read_table("metadata/performance_acquisition_universe.csv")
category_crosswalk = read_table("metadata/category_crosswalk.csv")
brand_crosswalk = read_table("metadata/brand_label_crosswalk.csv")
comparability_rules = read_table("metadata/comparability_rules.csv")
stage2_mapping_validation = read_table("metadata/stage2_mapping_validation.csv")
stage2_standardization_validation = read_table("metadata/stage2_standardization_validation.csv")
performance_observations = read_table("data/standardized/performance_observations.csv")

for validation_name, validation_table in [
    ("Stage 2 mapping", stage2_mapping_validation),
    ("Stage 2 standardization", stage2_standardization_validation),
]:
    critical_count = validation_table["critical_failure"].eq("yes").sum()
    if critical_count:
        raise AssertionError(f"{validation_name} contains {critical_count} critical failure(s).")

loaded_counts = pd.DataFrame(
    [
        ("ownership_registry", len(ownership)),
        ("performance_acquisition_universe", len(acquisition_universe)),
        ("category_crosswalk", len(category_crosswalk)),
        ("brand_label_crosswalk", len(brand_crosswalk)),
        ("comparability_rules", len(comparability_rules)),
        ("stage2_mapping_validation", len(stage2_mapping_validation)),
        ("stage2_standardization_validation", len(stage2_standardization_validation)),
        ("performance_observations", len(performance_observations)),
    ],
    columns=["table", "row_count"],
)
print(loaded_counts.to_string(index=False))


                            table  row_count
               ownership_registry        157
 performance_acquisition_universe         28
               category_crosswalk         43
            brand_label_crosswalk         31
              comparability_rules         18
        stage2_mapping_validation         18
stage2_standardization_validation         24
         performance_observations        100


## Structural Portfolio Universe

Retain every ownership record and add deterministic strict-control, current-ownership, category-mapping, and counting-eligibility fields. Non-primary, historical, joint-venture, affiliate, and context records remain visible.


In [5]:
ownership_category_map = (
    category_crosswalk.loc[
        category_crosswalk["input_domain"].eq("ownership_registry"),
        [
            "crosswalk_id",
            "source_category",
            "canonical_sector",
            "canonical_category",
            "canonical_subcategory",
            "primary_universe_status",
            "mapping_status",
        ],
    ]
    .rename(
        columns={
            "crosswalk_id": "category_crosswalk_id",
            "source_category": "category_scope",
            "mapping_status": "category_mapping_status",
        }
    )
)

if ownership_category_map["category_scope"].duplicated().any():
    raise ValueError("Ownership category crosswalk keys are not unique.")

structural_portfolio_universe = ownership.merge(
    ownership_category_map,
    on="category_scope",
    how="left",
    validate="many_to_one",
)
structural_portfolio_universe.insert(
    0,
    "structural_record_id",
    [f"STR_{index:03d}" for index in range(1, len(structural_portfolio_universe) + 1)],
)

reference_timestamp = pd.Timestamp(PREPARATION_REFERENCE_DATE)
ownership_end_parsed = pd.to_datetime(
    structural_portfolio_universe["ownership_end"],
    errors="coerce",
)
structural_portfolio_universe["current_at_reference_date"] = (
    ownership_end_parsed.isna() | ownership_end_parsed.ge(reference_timestamp)
).map({True: "yes", False: "no"})

strict_control_condition = (
    structural_portfolio_universe["relationship_type"].isin(PRIMARY_RELATIONSHIPS)
    & structural_portfolio_universe["current_at_reference_date"].eq("yes")
)
structural_portfolio_universe["strict_control_primary"] = strict_control_condition.map(
    {True: "yes", False: "no"}
)

normalized_group = (
    structural_portfolio_universe["group"]
    .str.casefold()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
normalized_family = (
    structural_portfolio_universe["brand_family"]
    .str.casefold()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
structural_portfolio_universe["structural_brand_count_key"] = (
    normalized_group + "__" + normalized_family
)

brand_breadth_eligible = strict_control_condition & structural_portfolio_universe[
    "brand_family"
].ne("")
category_breadth_eligible = (
    brand_breadth_eligible
    & structural_portfolio_universe["category_mapping_status"].isin(
        ELIGIBLE_CATEGORY_MAPPINGS
    )
    & structural_portfolio_universe["primary_universe_status"].isin(
        {"included", "included_with_caveat"}
    )
    & structural_portfolio_universe["canonical_category"].ne("")
)
structural_portfolio_universe["structural_brand_breadth_eligible"] = (
    brand_breadth_eligible.map({True: "yes", False: "no"})
)
structural_portfolio_universe["structural_category_breadth_eligible"] = (
    category_breadth_eligible.map({True: "yes", False: "no"})
)
structural_portfolio_universe["structural_category_count_key"] = ""
structural_portfolio_universe.loc[
    category_breadth_eligible, "structural_category_count_key"
] = (
    normalized_group[category_breadth_eligible]
    + "__"
    + structural_portfolio_universe.loc[
        category_breadth_eligible, "canonical_category"
    ]
    .str.casefold()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

structural_portfolio_universe["structural_exclusion_reason"] = ""
structural_portfolio_universe.loc[
    ~strict_control_condition, "structural_exclusion_reason"
] = "outside_current_strict_control_primary_view"
structural_portfolio_universe.loc[
    strict_control_condition
    & ~structural_portfolio_universe["category_mapping_status"].isin(
        ELIGIBLE_CATEGORY_MAPPINGS
    ),
    "structural_exclusion_reason",
] = "category_mapping_not_ready_for_category_breadth"
structural_portfolio_universe.loc[
    strict_control_condition
    & structural_portfolio_universe["category_mapping_status"].isin(
        ELIGIBLE_CATEGORY_MAPPINGS
    )
    & structural_portfolio_universe["canonical_category"].eq(""),
    "structural_exclusion_reason",
] = "canonical_category_unavailable"

structural_summary = (
    structural_portfolio_universe.groupby(
        ["group", "strict_control_primary", "category_mapping_status"],
        dropna=False,
    )
    .size()
    .rename("record_count")
    .reset_index()
)
print(structural_summary.to_string(index=False))


             group strict_control_primary      category_mapping_status  record_count
          Indofood                    yes                       direct            39
            Mayora                     no                       direct             1
            Mayora                    yes                       direct            24
Unilever Indonesia                     no                       direct             5
Unilever Indonesia                     no        excluded_context_only             3
Unilever Indonesia                    yes                       direct            19
Unilever Indonesia                    yes requires_brand_level_mapping             1
Unilever Indonesia                    yes         resolved_with_caveat             5
       Wings Group                     no                       direct             7
       Wings Group                     no                   unresolved             8
       Wings Group                    yes                   unres

## Competitive Observation Universe

Retain all standardized observations and add ownership-support, primary-scope, sensitivity-scope, category-comparison, and longitudinal-series fields. Records with unresolved ownership support remain visible but are blocked from primary calculations.


In [6]:
primary_ownership = structural_portfolio_universe[
    structural_portfolio_universe["strict_control_primary"].eq("yes")
].copy()
primary_ownership_keys = set(
    zip(
        primary_ownership["group"].str.casefold(),
        primary_ownership["brand_family"].str.casefold(),
    )
)

competitive_observation_universe = performance_observations.copy()
ownership_key_matches = [
    (group.casefold(), brand_family.casefold()) in primary_ownership_keys
    for group, brand_family in zip(
        competitive_observation_universe["canonical_group"],
        competitive_observation_universe["canonical_brand_family"],
    )
]
competitive_observation_universe["ownership_registry_primary_match"] = pd.Series(
    ownership_key_matches, index=competitive_observation_universe.index
).map({True: "yes", False: "no"})

primary_eligible = (
    competitive_observation_universe["attribution_scope"].eq("strict_control")
    & competitive_observation_universe["relationship_type"].isin(PRIMARY_RELATIONSHIPS)
    & competitive_observation_universe["ownership_period_status"].eq("valid")
    & competitive_observation_universe["ownership_registry_primary_match"].eq("yes")
)
sensitivity_eligible = competitive_observation_universe["attribution_scope"].eq(
    "extended_group_sensitivity_only"
)
competitive_observation_universe["primary_analysis_eligible"] = primary_eligible.map(
    {True: "yes", False: "no"}
)
competitive_observation_universe["sensitivity_analysis_eligible"] = (
    sensitivity_eligible.map({True: "yes", False: "no"})
)
competitive_observation_universe["analysis_scope"] = "context_or_pending"
competitive_observation_universe.loc[
    primary_eligible, "analysis_scope"
] = "primary_strict_control"
competitive_observation_universe.loc[
    sensitivity_eligible, "analysis_scope"
] = "extended_group_sensitivity_only"

available_value = competitive_observation_universe["observation_status"].isin(
    {"observed", "observed_lower_bound"}
)
category_comparison_candidate = (
    primary_eligible
    & competitive_observation_universe["source_family"].eq("Top Brand")
    & competitive_observation_universe["metric"].eq("TBI")
    & competitive_observation_universe["unit"].eq("percent_index_points")
    & competitive_observation_universe["observation_status"].eq("observed")
    & competitive_observation_universe["comparability_class"].eq(
        "comparable_with_caveat"
    )
    & competitive_observation_universe["source_subcategory_id"].ne("")
)
competitive_observation_universe["value_available"] = available_value.map(
    {True: "yes", False: "no"}
)
competitive_observation_universe["category_strength_candidate"] = (
    category_comparison_candidate.map({True: "yes", False: "no"})
)
competitive_observation_universe["category_leadership_value_candidate"] = (
    category_comparison_candidate.map({True: "yes", False: "no"})
)
competitive_observation_universe["consumer_reach_group_comparison_eligible"] = "no"

universe_attributes = acquisition_universe[
    [
        "universe_id",
        "acquisition_role",
        "target_period",
        "selection_basis",
        "capture_status",
        "capture_note",
    ]
].rename(columns={"target_period": "universe_target_period"})
competitive_observation_universe = competitive_observation_universe.merge(
    universe_attributes,
    on="universe_id",
    how="left",
    validate="many_to_one",
)

series_components = [
    competitive_observation_universe["universe_id"],
    competitive_observation_universe["source_brand_label"],
    competitive_observation_universe["source_subcategory_id"],
    competitive_observation_universe["metric"],
    competitive_observation_universe["unit"],
    competitive_observation_universe["geography"],
    competitive_observation_universe["methodology_cluster"],
]
competitive_observation_universe["source_label_series_key"] = series_components[0]
for component in series_components[1:]:
    competitive_observation_universe["source_label_series_key"] += "|" + component

family_components = [
    competitive_observation_universe["canonical_brand_key"],
    competitive_observation_universe["source_subcategory_id"],
    competitive_observation_universe["metric"],
    competitive_observation_universe["unit"],
    competitive_observation_universe["geography"],
    competitive_observation_universe["methodology_cluster"],
]
competitive_observation_universe["family_series_key"] = family_components[0]
for component in family_components[1:]:
    competitive_observation_universe["family_series_key"] += "|" + component

competitive_observation_universe["eligibility_exclusion_reason"] = ""
competitive_observation_universe.loc[
    competitive_observation_universe["attribution_scope"].eq("strict_control")
    & competitive_observation_universe["ownership_registry_primary_match"].eq("no"),
    "eligibility_exclusion_reason",
] = "strict_control_label_lacks_primary_ownership_registry_match"
competitive_observation_universe.loc[
    competitive_observation_universe["attribution_scope"].eq("strict_control")
    & ~competitive_observation_universe["relationship_type"].isin(
        PRIMARY_RELATIONSHIPS
    ),
    "eligibility_exclusion_reason",
] = "relationship_type_not_verified_for_primary_analysis"
competitive_observation_universe.loc[
    competitive_observation_universe["observation_status"].eq("not_available"),
    "eligibility_exclusion_reason",
] = "explicit_source_unavailability"
competitive_observation_universe.loc[
    sensitivity_eligible, "eligibility_exclusion_reason"
] = "retained_for_extended_group_sensitivity_only"
competitive_observation_universe.loc[
    competitive_observation_universe["comparability_class"].eq("context_only")
    & ~sensitivity_eligible,
    "eligibility_exclusion_reason",
] = "context_only_source_fact"

competitive_summary = (
    competitive_observation_universe.groupby(
        ["canonical_group", "analysis_scope", "observation_status"],
        dropna=False,
    )
    .size()
    .rename("observation_count")
    .reset_index()
)
print(competitive_summary.to_string(index=False))


   canonical_group                  analysis_scope   observation_status  observation_count
          Indofood              context_or_pending             observed                  5
          Indofood          primary_strict_control             observed                 22
            Mayora extended_group_sensitivity_only             observed                  2
            Mayora          primary_strict_control        not_available                  5
            Mayora          primary_strict_control             observed                  5
Unilever Indonesia          primary_strict_control        not_available                  5
Unilever Indonesia          primary_strict_control             observed                 18
       Wings Group          primary_strict_control             observed                 37
       Wings Group          primary_strict_control observed_lower_bound                  1


## Eligible and Observable Denominators

Expand the frozen acquisition universe into target-period members, reconcile each member with standardized observations, and aggregate group-category-period denominators. Eligible, observable, observed, unavailable, not-observed, pending-ownership, context-only, and sensitivity members remain separate.


In [7]:
performance_by_universe = {
    universe_id: rows.copy()
    for universe_id, rows in competitive_observation_universe.groupby("universe_id")
}

performance_category_lookup = category_crosswalk[
    category_crosswalk["input_domain"].eq("source_native_performance")
][
    [
        "source_family",
        "source_subcategory_id",
        "canonical_sector",
        "canonical_category",
        "canonical_subcategory",
    ]
].drop_duplicates()

def expand_target_periods(universe_row: pd.Series) -> list[str]:
    target_period = universe_row["target_period"]
    if universe_row["source_family"] == "Top Brand":
        range_match = re.fullmatch(r"(\d{4})-(\d{4})", target_period)
        if range_match:
            start_year, end_year = map(int, range_match.groups())
            return [str(year) for year in range(start_year, end_year + 1)]
        if re.fullmatch(r"\d{4}", target_period):
            return [target_period]
        raise ValueError(
            f"Unsupported Top Brand target period for {universe_row['universe_id']}: {target_period}"
        )

    actual_rows = performance_by_universe.get(universe_row["universe_id"])
    if actual_rows is not None and not actual_rows.empty:
        return sorted(actual_rows["reference_period"].drop_duplicates().tolist())
    return [target_period]


def top_brand_methodology_cluster(reference_period: str) -> str:
    if reference_period in {"2022", "2023", "2024", "2025"}:
        return "top_brand_historical_unverified"
    if reference_period == "2026":
        return "top_brand_current_documented"
    return "not_applicable"


target_rows = []
for _, universe_row in acquisition_universe.iterrows():
    canonical_group = (
        "Mayora"
        if universe_row["group"] == "Mayora extended group"
        else universe_row["group"]
    )
    ownership_match = (
        canonical_group.casefold(),
        universe_row["brand_family"].casefold(),
    ) in primary_ownership_keys

    for reference_period in expand_target_periods(universe_row):
        actual_rows = competitive_observation_universe[
            competitive_observation_universe["universe_id"].eq(
                universe_row["universe_id"]
            )
            & competitive_observation_universe["reference_period"].eq(
                reference_period
            )
        ].copy()

        canonical_sector = ""
        canonical_category = ""
        canonical_subcategory = ""
        if universe_row["source_family"] == "Top Brand":
            mapping = performance_category_lookup[
                performance_category_lookup["source_family"].eq("Top Brand")
                & performance_category_lookup["source_subcategory_id"].eq(
                    universe_row["source_subcategory_id"]
                )
            ]
            if len(mapping) != 1:
                raise ValueError(
                    f"Expected one performance category mapping for {universe_row['universe_id']}."
                )
            canonical_sector = mapping.iloc[0]["canonical_sector"]
            canonical_category = mapping.iloc[0]["canonical_category"]
            canonical_subcategory = mapping.iloc[0]["canonical_subcategory"]
        elif not actual_rows.empty:
            for field in [
                "canonical_sector",
                "canonical_category",
                "canonical_subcategory",
            ]:
                values = actual_rows.loc[actual_rows[field].ne(""), field].unique()
                if len(values) == 1:
                    if field == "canonical_sector":
                        canonical_sector = values[0]
                    elif field == "canonical_category":
                        canonical_category = values[0]
                    else:
                        canonical_subcategory = values[0]

        observed_rows = actual_rows[
            actual_rows["observation_status"].isin(
                {"observed", "observed_lower_bound"}
            )
        ]
        observable_member = not actual_rows.empty
        observed_member = not observed_rows.empty
        explicit_not_available = (
            observable_member
            and not observed_member
            and actual_rows["observation_status"].eq("not_available").all()
        )
        context_only_member = (
            observable_member
            and actual_rows["comparability_class"].eq("context_only").any()
        )

        strict_scope = universe_row["attribution_scope"] == "strict_control"
        primary_ownership_valid = strict_scope and ownership_match
        pending_ownership = strict_scope and not ownership_match
        sensitivity_member = (
            universe_row["attribution_scope"]
            == "extended_group_sensitivity_only"
        )

        target_rows.append(
            {
                "universe_id": universe_row["universe_id"],
                "canonical_group": canonical_group,
                "source_family": universe_row["source_family"],
                "source_category": universe_row["source_category"],
                "source_subcategory": universe_row["source_subcategory"],
                "source_subcategory_id": universe_row["source_subcategory_id"],
                "canonical_sector": canonical_sector,
                "canonical_category": canonical_category,
                "canonical_subcategory": canonical_subcategory,
                "reference_period": reference_period,
                "methodology_cluster": (
                    top_brand_methodology_cluster(reference_period)
                    if universe_row["source_family"] == "Top Brand"
                    else (
                        actual_rows["methodology_cluster"].iloc[0]
                        if not actual_rows.empty
                        else "not_applicable"
                    )
                ),
                "metric_scope": (
                    "TBI"
                    if universe_row["source_family"] == "Top Brand"
                    else "limited_public_fact"
                ),
                "attribution_scope": universe_row["attribution_scope"],
                "eligible_target_count": int(primary_ownership_valid),
                "pending_ownership_target_count": int(pending_ownership),
                "sensitivity_target_count": int(sensitivity_member),
                "observable_target_count": int(observable_member),
                "observed_target_count": int(observed_member),
                "explicit_not_available_target_count": int(
                    explicit_not_available
                ),
                "not_observed_target_count": int(not observable_member),
                "context_only_target_count": int(context_only_member),
                "standardized_observation_row_count": len(actual_rows),
            }
        )

denominator_targets = pd.DataFrame(target_rows)
denominator_group_keys = [
    "canonical_group",
    "source_family",
    "source_category",
    "source_subcategory",
    "source_subcategory_id",
    "canonical_sector",
    "canonical_category",
    "canonical_subcategory",
    "reference_period",
    "methodology_cluster",
    "metric_scope",
    "attribution_scope",
]
count_columns = [
    "eligible_target_count",
    "pending_ownership_target_count",
    "sensitivity_target_count",
    "observable_target_count",
    "observed_target_count",
    "explicit_not_available_target_count",
    "not_observed_target_count",
    "context_only_target_count",
    "standardized_observation_row_count",
]
group_category_period_denominators = (
    denominator_targets.groupby(denominator_group_keys, dropna=False)[count_columns]
    .sum()
    .reset_index()
)

def group_denominator_status(row: pd.Series) -> str:
    if row["sensitivity_target_count"] > 0:
        return "sensitivity_only"
    if row["source_family"] == "Brand Footprint":
        return "limited_public_facts_context"
    if row["pending_ownership_target_count"] > 0:
        return "blocked_pending_ownership"
    if row["not_observed_target_count"] > 0:
        return "incomplete_not_observed"
    if row["explicit_not_available_target_count"] > 0:
        return "observable_with_explicit_unavailability"
    if row["observed_target_count"] == row["eligible_target_count"]:
        return "complete_observed"
    return "not_eligible"

group_category_period_denominators["group_denominator_status"] = (
    group_category_period_denominators.apply(group_denominator_status, axis=1)
)

category_total_keys = [
    "source_family",
    "source_category",
    "source_subcategory",
    "source_subcategory_id",
    "canonical_sector",
    "canonical_category",
    "canonical_subcategory",
    "reference_period",
    "methodology_cluster",
    "metric_scope",
]
strict_targets = denominator_targets[
    denominator_targets["attribution_scope"].eq("strict_control")
].copy()
category_totals = (
    strict_targets.groupby(category_total_keys, dropna=False)
    .agg(
        category_eligible_target_count=("eligible_target_count", "sum"),
        category_pending_ownership_target_count=(
            "pending_ownership_target_count",
            "sum",
        ),
        category_observable_target_count=("observable_target_count", "sum"),
        category_observed_target_count=("observed_target_count", "sum"),
        category_explicit_not_available_target_count=(
            "explicit_not_available_target_count",
            "sum",
        ),
        category_not_observed_target_count=("not_observed_target_count", "sum"),
        category_distinct_eligible_groups=(
            "canonical_group",
            lambda values: values[
                strict_targets.loc[values.index, "eligible_target_count"].eq(1)
            ].nunique(),
        ),
    )
    .reset_index()
)

def category_denominator_status(row: pd.Series) -> str:
    if row["source_family"] == "Brand Footprint":
        return "limited_public_facts_context"
    if row["category_pending_ownership_target_count"] > 0:
        return "blocked_pending_ownership"
    if row["category_distinct_eligible_groups"] < 2:
        return "not_eligible_single_group"
    if row["category_not_observed_target_count"] > 0:
        return "incomplete_not_observed"
    if row["category_explicit_not_available_target_count"] > 0:
        return "observable_with_explicit_unavailability"
    if (
        row["category_observed_target_count"]
        == row["category_eligible_target_count"]
    ):
        return "complete_observed"
    return "not_eligible"

category_totals["category_denominator_status"] = category_totals.apply(
    category_denominator_status, axis=1
)
group_category_period_denominators = group_category_period_denominators.merge(
    category_totals,
    on=category_total_keys,
    how="left",
    validate="many_to_one",
)

print(
    group_category_period_denominators[
        [
            "canonical_group",
            "source_family",
            "canonical_subcategory",
            "reference_period",
            "eligible_target_count",
            "pending_ownership_target_count",
            "observed_target_count",
            "not_observed_target_count",
            "group_denominator_status",
            "category_denominator_status",
        ]
    ].to_string(index=False)
)


   canonical_group   source_family       canonical_subcategory   reference_period  eligible_target_count  pending_ownership_target_count  observed_target_count  not_observed_target_count                group_denominator_status             category_denominator_status
          Indofood Brand Footprint             Instant Noodles               2020                      1                               0                      1                          0            limited_public_facts_context            limited_public_facts_context
          Indofood Brand Footprint             Food Seasonings               2020                      1                               0                      1                          0            limited_public_facts_context            limited_public_facts_context
          Indofood       Top Brand             Sweet Soy Sauce               2022                      1                               0                      1                          0             

## Dimension-Level Eligibility Rules

Register the pre-specified requirements, denominator treatment, and current preparation status for every analytical dimension. These rules are set before portfolio-performance calculations.


In [8]:
stage3_metric_eligibility_rules = pd.DataFrame(
    [
        {
            "dimension_id": "DIM01A",
            "dimension": "structural_brand_breadth",
            "eligible_source": "ownership_registry",
            "required_metric_or_unit": "current strict-control canonical brand-family count",
            "minimum_observations_or_periods": "not_applicable",
            "denominator_rule": "All current controlled and controlled_group_portfolio ownership records; deduplicate by group and brand family.",
            "current_eligibility": "eligible_with_caveat",
            "prohibited_use": "Do not substitute survey-observed brands for the structural ownership universe.",
            "eligibility_reason": "Brand-family counting is possible, but unresolved category assignments remain visible.",
        },
        {
            "dimension_id": "DIM01B",
            "dimension": "structural_category_breadth",
            "eligible_source": "ownership_registry plus category_crosswalk",
            "required_metric_or_unit": "mapped canonical categories",
            "minimum_observations_or_periods": "complete comparable category mapping by focal group",
            "denominator_rule": "Count a brand family once within each defensibly mapped canonical category.",
            "current_eligibility": "not_eligible_currently",
            "prohibited_use": "Do not infer ownership categories from survey visibility.",
            "eligibility_reason": "Wings Group ownership categories remain unresolved in the committed structural taxonomy.",
        },
        {
            "dimension_id": "DIM02",
            "dimension": "competitive_breadth",
            "eligible_source": "frozen performance_acquisition_universe",
            "required_metric_or_unit": "eligible and observable denominator counts",
            "minimum_observations_or_periods": "one pre-specified target member",
            "denominator_rule": "Report eligible, observable, observed, unavailable, and not-observed members separately.",
            "current_eligibility": "eligible_with_caveat",
            "prohibited_use": "Do not treat survey omission as zero or structural absence.",
            "eligibility_reason": "Selective public coverage supports only denominator-explicit competitive breadth.",
        },
        {
            "dimension_id": "DIM03",
            "dimension": "consumer_reach",
            "eligible_source": "Brand Footprint",
            "required_metric_or_unit": "comparable CRP or equivalent source-native reach universe",
            "minimum_observations_or_periods": "comparable focal-group observations in the same edition and geography cluster",
            "denominator_rule": "Use only source-native comparable consumer-reach measurements.",
            "current_eligibility": "not_eligible_currently",
            "prohibited_use": "Do not treat rank, lower bounds, qualitative facts, or public-summary omission as CRP or market share.",
            "eligibility_reason": "Current limited public facts do not provide a comparable CRP universe across focal groups.",
        },
        {
            "dimension_id": "DIM04",
            "dimension": "category_strength",
            "eligible_source": "Top Brand",
            "required_metric_or_unit": "TBI; percent_index_points",
            "minimum_observations_or_periods": "one observed strict-control value",
            "denominator_rule": "Compare only within the same source subcategory, reference period, geography, unit, and methodology cluster.",
            "current_eligibility": "eligible_with_caveat",
            "prohibited_use": "Do not pool or average TBI across categories or relabel TBI as market share.",
            "eligibility_reason": "Category-period comparison is possible where ownership and observation eligibility pass.",
        },
        {
            "dimension_id": "DIM05",
            "dimension": "category_leadership",
            "eligible_source": "Top Brand",
            "required_metric_or_unit": "TBI; ordinal focal-leader result",
            "minimum_observations_or_periods": "at least two focal groups and a complete observed category denominator",
            "denominator_rule": "Identify only the highest observed focal-group TBI within an eligible category-period universe.",
            "current_eligibility": "eligible_conditionally",
            "prohibited_use": "Do not call a focal-group leader the full market category leader.",
            "eligibility_reason": "Leadership is permitted only when every eligible pre-specified focal target is observed.",
        },
        {
            "dimension_id": "DIM06",
            "dimension": "longitudinal_consistency",
            "eligible_source": "Top Brand",
            "required_metric_or_unit": "TBI; percent_index_points",
            "minimum_observations_or_periods": "three consecutive observed periods",
            "denominator_rule": "Use one source-label series within one category, geography, metric, unit, and methodology cluster.",
            "current_eligibility": "eligible_conditionally",
            "prohibited_use": "Do not bridge unresolved label, category, ownership, or methodology breaks.",
            "eligibility_reason": "Eligible series are determined individually before consistency calculations.",
        },
        {
            "dimension_id": "DIM07",
            "dimension": "competitive_persistence",
            "eligible_source": "Top Brand",
            "required_metric_or_unit": "TBI baseline incumbent and comparable follow-up periods",
            "minimum_observations_or_periods": "complete baseline plus at least two complete follow-up periods",
            "denominator_rule": "Baseline incumbent is the highest focal-group TBI in the earliest complete common period within one category and methodology cluster.",
            "current_eligibility": "eligible_conditionally",
            "prohibited_use": "Do not define an incumbent from an incomplete, single-group, or ownership-pending denominator.",
            "eligibility_reason": "Persistence calculations are deferred until baseline and follow-up eligibility both pass.",
        },
        {
            "dimension_id": "DIM08",
            "dimension": "portfolio_concentration",
            "eligible_source": "dimension-level analytical outputs",
            "required_metric_or_unit": "additive and sufficiently complete performance universe",
            "minimum_observations_or_periods": "complete comparable portfolio distribution",
            "denominator_rule": "Assess performance concentration only after a defensible additive universe exists.",
            "current_eligibility": "not_eligible_currently",
            "prohibited_use": "Do not calculate HHI from cross-category TBI, ranks, lower bounds, or selective public facts.",
            "eligibility_reason": "Current source-native metrics are not an additive, sufficiently complete portfolio-performance universe.",
        },
        {
            "dimension_id": "DIM09",
            "dimension": "momentum",
            "eligible_source": "Top Brand",
            "required_metric_or_unit": "TBI; percent_index_points",
            "minimum_observations_or_periods": "three consecutive observed periods providing at least two intervals",
            "denominator_rule": "Use one eligible source-label series within a single methodology cluster.",
            "current_eligibility": "eligible_conditionally",
            "prohibited_use": "Do not calculate ordinary movement across the 2025–2026 methodology-status boundary.",
            "eligibility_reason": "Eligible series are screened before any change calculation.",
        },
        {
            "dimension_id": "DIM10",
            "dimension": "overall_portfolio_leadership",
            "eligible_source": "validated dimension-level results",
            "required_metric_or_unit": "pre-specified transparent synthesis with sensitivity testing",
            "minimum_observations_or_periods": "all material dimension and denominator gates resolved",
            "denominator_rule": "Defer overall synthesis until dimension-specific evidence is complete and compatible.",
            "current_eligibility": "deferred",
            "prohibited_use": "Do not create a composite score or select weights after observing winners.",
            "eligibility_reason": "Stage 3A prepares eligibility only and does not determine an overall winner.",
        },
    ]
)
print(
    stage3_metric_eligibility_rules[
        ["dimension_id", "dimension", "current_eligibility"]
    ].to_string(index=False)
)


dimension_id                    dimension    current_eligibility
      DIM01A     structural_brand_breadth   eligible_with_caveat
      DIM01B  structural_category_breadth not_eligible_currently
       DIM02          competitive_breadth   eligible_with_caveat
       DIM03               consumer_reach not_eligible_currently
       DIM04            category_strength   eligible_with_caveat
       DIM05          category_leadership eligible_conditionally
       DIM06     longitudinal_consistency eligible_conditionally
       DIM07      competitive_persistence eligible_conditionally
       DIM08      portfolio_concentration not_eligible_currently
       DIM09                     momentum eligible_conditionally
       DIM10 overall_portfolio_leadership               deferred


## Longitudinal Series Eligibility

Evaluate each source-label-specific Top Brand series without crossing category, ownership, geography, metric, unit, label, or methodology boundaries. A minimum of three consecutive observed periods is required for consistency and momentum.


In [9]:
def maximum_consecutive_run(periods: list[int]) -> int:
    if not periods:
        return 0
    ordered = sorted(set(periods))
    longest = 1
    current = 1
    for previous, current_period in zip(ordered, ordered[1:]):
        if current_period == previous + 1:
            current += 1
            longest = max(longest, current)
        else:
            current = 1
    return longest


top_brand_series_rows = competitive_observation_universe[
    competitive_observation_universe["source_family"].eq("Top Brand")
    & competitive_observation_universe["metric"].eq("TBI")
    & competitive_observation_universe["unit"].eq("percent_index_points")
].copy()

series_rows = []
for series_key, rows in top_brand_series_rows.groupby(
    "source_label_series_key", sort=True
):
    observed = rows[rows["observation_status"].eq("observed")].copy()
    observed_periods = sorted(
        int(period)
        for period in observed["reference_period"].unique()
        if re.fullmatch(r"\d{4}", period)
    )
    maximum_run = maximum_consecutive_run(observed_periods)
    ownership_valid = rows["primary_analysis_eligible"].eq("yes").all()
    minimum_period_gate = len(observed_periods) >= 3 and maximum_run >= 3
    alias_review = rows["acquisition_role"].eq(
        "longitudinal_alias_review"
    ).any()

    if not ownership_valid:
        consistency_status = "not_eligible_pending_ownership"
        momentum_status = "not_eligible_pending_ownership"
        exclusion_reason = "primary ownership support is unresolved"
    elif not minimum_period_gate:
        consistency_status = "not_eligible_insufficient_periods"
        momentum_status = "not_eligible_insufficient_periods"
        exclusion_reason = "fewer than three consecutive observed periods"
    else:
        consistency_status = "eligible_with_caveat"
        momentum_status = "eligible_with_caveat"
        exclusion_reason = ""

    representative = rows.iloc[0]
    series_rows.append(
        {
            "source_label_series_key": series_key,
            "family_series_key": representative["family_series_key"],
            "universe_id": representative["universe_id"],
            "canonical_group": representative["canonical_group"],
            "canonical_brand_family": representative["canonical_brand_family"],
            "canonical_brand_key": representative["canonical_brand_key"],
            "source_brand_label": representative["source_brand_label"],
            "canonical_sector": representative["canonical_sector"],
            "canonical_category": representative["canonical_category"],
            "canonical_subcategory": representative["canonical_subcategory"],
            "source_subcategory_id": representative[
                "source_subcategory_id"
            ],
            "metric": representative["metric"],
            "unit": representative["unit"],
            "geography": representative["geography"],
            "methodology_cluster": representative["methodology_cluster"],
            "methodology_status": representative["methodology_status"],
            "attribution_scope": representative["attribution_scope"],
            "ownership_support_status": (
                "verified_primary" if ownership_valid else "pending"
            ),
            "identity_continuity_treatment": (
                "source_label_segment_only"
                if alias_review
                else "single_source_label"
            ),
            "available_period_count": len(rows),
            "observed_period_count": len(observed_periods),
            "first_observed_period": (
                str(observed_periods[0]) if observed_periods else ""
            ),
            "last_observed_period": (
                str(observed_periods[-1]) if observed_periods else ""
            ),
            "observed_periods": ";".join(map(str, observed_periods)),
            "maximum_consecutive_observed_periods": maximum_run,
            "consistency_eligibility": consistency_status,
            "momentum_eligibility": momentum_status,
            "persistence_follow_up_series_candidate": (
                "yes"
                if ownership_valid and minimum_period_gate
                else "no"
            ),
            "eligibility_exclusion_reason": exclusion_reason,
            "methodology_caveat": (
                "Historical Top Brand methodology is not independently verified."
                if representative["methodology_cluster"]
                == "top_brand_historical_unverified"
                else "Current documented cluster is not joined to the historical cluster."
            ),
        }
    )

longitudinal_series_eligibility = pd.DataFrame(series_rows)
print(
    longitudinal_series_eligibility[
        [
            "canonical_group",
            "canonical_brand_family",
            "source_brand_label",
            "canonical_subcategory",
            "methodology_cluster",
            "observed_period_count",
            "consistency_eligibility",
            "momentum_eligibility",
        ]
    ].to_string(index=False)
)


   canonical_group canonical_brand_family source_brand_label       canonical_subcategory             methodology_cluster  observed_period_count           consistency_eligibility              momentum_eligibility
       Wings Group                 Sedaap         Mie Sedaap      Bagged Instant Noodles    top_brand_current_documented                      1 not_eligible_insufficient_periods not_eligible_insufficient_periods
       Wings Group                 Sedaap         Mie Sedaap      Bagged Instant Noodles top_brand_historical_unverified                      4              eligible_with_caveat              eligible_with_caveat
          Indofood                Indomie            Indomie      Bagged Instant Noodles    top_brand_current_documented                      1 not_eligible_insufficient_periods not_eligible_insufficient_periods
          Indofood                Indomie            Indomie      Bagged Instant Noodles top_brand_historical_unverified                      4         

## Persistence Baseline Candidates

Identify the earliest complete common period within each Top Brand category and methodology cluster. Select the highest focal-group TBI only as a baseline incumbent candidate; do not calculate persistence outcomes.


In [10]:
observed_primary_tbi = competitive_observation_universe[
    competitive_observation_universe["category_strength_candidate"].eq("yes")
].copy()
observed_primary_tbi["value_numeric"] = pd.to_numeric(
    observed_primary_tbi["value"], errors="raise"
)

duplicate_observed_targets = observed_primary_tbi.duplicated(
    ["universe_id", "reference_period"], keep=False
)
if duplicate_observed_targets.any():
    raise ValueError(
        "More than one observed value exists for the same universe target and period."
    )

persistence_rows = []
top_brand_target_categories = (
    denominator_targets[
        denominator_targets["source_family"].eq("Top Brand")
        & denominator_targets["attribution_scope"].eq("strict_control")
    ][
        [
            "source_category",
            "source_subcategory",
            "source_subcategory_id",
            "canonical_sector",
            "canonical_category",
            "canonical_subcategory",
            "methodology_cluster",
        ]
    ]
    .drop_duplicates()
    .sort_values(["source_subcategory_id", "methodology_cluster"])
)

for _, category_row in top_brand_target_categories.iterrows():
    category_targets = denominator_targets[
        denominator_targets["source_family"].eq("Top Brand")
        & denominator_targets["attribution_scope"].eq("strict_control")
        & denominator_targets["source_subcategory_id"].eq(
            category_row["source_subcategory_id"]
        )
        & denominator_targets["methodology_cluster"].eq(
            category_row["methodology_cluster"]
        )
    ].copy()

    period_status_rows = []
    for reference_period, period_targets in category_targets.groupby(
        "reference_period", sort=True
    ):
        eligible_targets = period_targets[
            period_targets["eligible_target_count"].eq(1)
        ]
        period_status_rows.append(
            {
                "reference_period": reference_period,
                "eligible_target_count": int(
                    eligible_targets["eligible_target_count"].sum()
                ),
                "pending_ownership_target_count": int(
                    period_targets["pending_ownership_target_count"].sum()
                ),
                "observed_target_count": int(
                    eligible_targets["observed_target_count"].sum()
                ),
                "not_observed_target_count": int(
                    eligible_targets["not_observed_target_count"].sum()
                ),
                "explicit_not_available_target_count": int(
                    eligible_targets[
                        "explicit_not_available_target_count"
                    ].sum()
                ),
                "distinct_eligible_groups": eligible_targets[
                    "canonical_group"
                ].nunique(),
            }
        )
    period_status = pd.DataFrame(period_status_rows)
    period_status["complete_common_period"] = (
        period_status["pending_ownership_target_count"].eq(0)
        & period_status["distinct_eligible_groups"].ge(2)
        & period_status["not_observed_target_count"].eq(0)
        & period_status["explicit_not_available_target_count"].eq(0)
        & period_status["observed_target_count"].eq(
            period_status["eligible_target_count"]
        )
    )

    complete_periods = period_status[
        period_status["complete_common_period"]
    ]["reference_period"].sort_values().tolist()

    base_output = {
        "source_family": "Top Brand",
        "source_category": category_row["source_category"],
        "source_subcategory": category_row["source_subcategory"],
        "source_subcategory_id": category_row["source_subcategory_id"],
        "canonical_sector": category_row["canonical_sector"],
        "canonical_category": category_row["canonical_category"],
        "canonical_subcategory": category_row["canonical_subcategory"],
        "methodology_cluster": category_row["methodology_cluster"],
        "baseline_rule": "Highest observed focal-group TBI in the earliest complete common period.",
    }

    if not complete_periods:
        persistence_rows.append(
            {
                **base_output,
                "baseline_status": "not_identified",
                "baseline_period": "",
                "baseline_incumbent_group": "",
                "baseline_incumbent_brand": "",
                "baseline_incumbent_observation_id": "",
                "baseline_tbi": "",
                "baseline_tie_count": "0",
                "complete_follow_up_period_count": "0",
                "complete_follow_up_periods": "",
                "persistence_analysis_eligibility": "not_eligible",
                "eligibility_reason": "No complete common period satisfies ownership, coverage, and multi-group gates.",
            }
        )
        continue

    baseline_period = complete_periods[0]
    baseline_candidates = observed_primary_tbi[
        observed_primary_tbi["source_subcategory_id"].eq(
            category_row["source_subcategory_id"]
        )
        & observed_primary_tbi["methodology_cluster"].eq(
            category_row["methodology_cluster"]
        )
        & observed_primary_tbi["reference_period"].eq(baseline_period)
    ].copy()
    maximum_tbi = baseline_candidates["value_numeric"].max()
    incumbent_candidates = baseline_candidates[
        baseline_candidates["value_numeric"].eq(maximum_tbi)
    ].sort_values(["canonical_group", "canonical_brand_family"])
    incumbent = incumbent_candidates.iloc[0]
    follow_up_periods = [
        period for period in complete_periods if period > baseline_period
    ]
    tie_count = len(incumbent_candidates)
    persistence_eligible = tie_count == 1 and len(follow_up_periods) >= 2

    persistence_rows.append(
        {
            **base_output,
            "baseline_status": (
                "identified" if tie_count == 1 else "ambiguous_tie"
            ),
            "baseline_period": baseline_period,
            "baseline_incumbent_group": incumbent["canonical_group"],
            "baseline_incumbent_brand": incumbent[
                "canonical_brand_family"
            ],
            "baseline_incumbent_observation_id": incumbent["observation_id"],
            "baseline_tbi": incumbent["value"],
            "baseline_tie_count": str(tie_count),
            "complete_follow_up_period_count": str(len(follow_up_periods)),
            "complete_follow_up_periods": ";".join(follow_up_periods),
            "persistence_analysis_eligibility": (
                "eligible_with_caveat"
                if persistence_eligible
                else "not_eligible"
            ),
            "eligibility_reason": (
                "Baseline and at least two complete follow-up periods are available within one methodology cluster."
                if persistence_eligible
                else (
                    "Baseline incumbent is tied."
                    if tie_count > 1
                    else "Fewer than two complete follow-up periods are available within the methodology cluster."
                )
            ),
        }
    )

persistence_baseline_candidates = pd.DataFrame(persistence_rows)
print(
    persistence_baseline_candidates[
        [
            "canonical_subcategory",
            "methodology_cluster",
            "baseline_status",
            "baseline_period",
            "baseline_incumbent_group",
            "baseline_incumbent_brand",
            "complete_follow_up_period_count",
            "persistence_analysis_eligibility",
        ]
    ].to_string(index=False)
)


      canonical_subcategory             methodology_cluster baseline_status baseline_period baseline_incumbent_group baseline_incumbent_brand complete_follow_up_period_count persistence_analysis_eligibility
               White Coffee    top_brand_current_documented      identified            2026              Wings Group               TOP Coffee                               0                     not_eligible
               White Coffee top_brand_historical_unverified      identified            2025              Wings Group               TOP Coffee                               0                     not_eligible
Ready-to-Drink Fruit Drinks    top_brand_current_documented      identified            2026       Unilever Indonesia                  Buavita                               0                     not_eligible
Ready-to-Drink Fruit Drinks top_brand_historical_unverified      identified            2022       Unilever Indonesia                  Buavita                               

## Stage 3A Validation

Validate input preservation, ownership attribution, denominator reconciliation, sensitivity isolation, explicit missingness, longitudinal boundaries, metric separation, and the absence of winner or composite-score outputs.


In [11]:
validation_rows = []

def add_check(
    check_id: str,
    area: str,
    description: str,
    condition: bool,
    result: str,
    critical: bool,
    treatment: str,
    notes: str = "",
    force_caveat: bool = False,
) -> None:
    if condition:
        status = "passed_with_caveat" if force_caveat else "passed"
        critical_failure = "no"
    else:
        status = "failed" if critical else "passed_with_caveat"
        critical_failure = "yes" if critical else "no"
    validation_rows.append(
        {
            "check_id": check_id,
            "validation_area": area,
            "check_description": description,
            "result": result,
            "status": status,
            "critical_failure": critical_failure,
            "required_treatment": treatment,
            "notes": notes,
        }
    )

add_check("S3A001", "input_integrity", "All Stage 3A inputs match the locked repository checksums", checksum_validation["status"].eq("passed").all(), f"{checksum_validation['status'].eq('passed').sum()}/{len(checksum_validation)} passed", True, "Stop preparation when an input checksum differs")
add_check("S3A002", "prior_stage_gate", "Stage 2 mapping and standardization validations contain no critical failures", not stage2_mapping_validation["critical_failure"].eq("yes").any() and not stage2_standardization_validation["critical_failure"].eq("yes").any(), "0 prior-stage critical failures", True, "Resolve prior-stage critical failures before preparation")
add_check("S3A003", "ownership_preservation", "All ownership records are retained in the structural universe", len(structural_portfolio_universe) == len(ownership) == 157, f"{len(structural_portfolio_universe)}/157 records retained", True, "Retain primary and non-primary ownership records")
add_check("S3A004", "strict_control_scope", "The structural universe retains 133 current strict-control primary records", structural_portfolio_universe["strict_control_primary"].eq("yes").sum() == 133, f"{structural_portfolio_universe['strict_control_primary'].eq('yes').sum()} primary records", True, "Use only controlled and controlled_group_portfolio current records for the primary structural view")
add_check("S3A005", "nonprimary_preservation", "All 24 non-primary, historical, joint-venture, affiliate, or context records remain visible", structural_portfolio_universe["strict_control_primary"].eq("no").sum() == 24, f"{structural_portfolio_universe['strict_control_primary'].eq('no').sum()} non-primary records", True, "Do not delete records outside the primary view")
add_check("S3A006", "acquisition_universe", "The frozen acquisition universe retains all 28 pre-specified members", len(acquisition_universe) == 28, f"{len(acquisition_universe)} universe members", True, "Do not select the observation universe after reviewing performance")
add_check("S3A007", "observation_preservation", "All 100 standardized observations are retained in the competitive universe", len(competitive_observation_universe) == len(performance_observations) == 100, f"{len(competitive_observation_universe)} observations retained", True, "Do not drop unavailable, context, or sensitivity observations")
add_check("S3A008", "observation_uniqueness", "Canonical observation identifiers remain unique", competitive_observation_universe["observation_id"].is_unique, f"{competitive_observation_universe['observation_id'].nunique()} unique IDs", True, "Resolve duplicate observation identifiers")
add_check("S3A009", "missingness", "Explicitly unavailable observations retain blank values rather than zero", competitive_observation_universe.loc[competitive_observation_universe["observation_status"].eq("not_available"), "value"].eq("").all(), f"{competitive_observation_universe['observation_status'].eq('not_available').sum()} blank unavailable values", True, "Never replace unavailable values with zero")
add_check("S3A010", "category_mapping", "Structural category breadth remains blocked where ownership categories are unresolved", structural_portfolio_universe.loc[structural_portfolio_universe["category_mapping_status"].eq("unresolved"), "structural_category_breadth_eligible"].eq("no").all(), f"{structural_portfolio_universe['category_mapping_status'].eq('unresolved').sum()} unresolved records excluded from category breadth", True, "Resolve ownership category mappings before category-breadth comparison", "Wings Group structural category mapping remains incomplete.", True)
pending_primary_observations = competitive_observation_universe["attribution_scope"].eq("strict_control") & competitive_observation_universe["primary_analysis_eligible"].eq("no") & competitive_observation_universe["ownership_registry_primary_match"].eq("no")
add_check("S3A011", "ownership_support", "Strict-control observations without primary ownership-registry support are blocked from primary calculations", competitive_observation_universe.loc[pending_primary_observations, "analysis_scope"].eq("context_or_pending").all(), f"{pending_primary_observations.sum()} observations blocked pending ownership support", True, "Add authoritative ownership support before primary use", "The current unmatched records are Indocafe observations.", True)
le_minerale = competitive_observation_universe["canonical_brand_family"].eq("Le Minerale")
add_check("S3A012", "sensitivity_isolation", "Le Minerale remains exclusively in the extended-group sensitivity view", le_minerale.sum() == 2 and competitive_observation_universe.loc[le_minerale, "sensitivity_analysis_eligible"].eq("yes").all() and competitive_observation_universe.loc[le_minerale, "primary_analysis_eligible"].eq("no").all(), f"{le_minerale.sum()} sensitivity-only observations", True, "Exclude Le Minerale from strict-control Mayora calculations")
add_check("S3A013", "denominator_reconciliation", "Expanded denominator targets reconcile to eligible, pending-ownership, and sensitivity members", (denominator_targets[["eligible_target_count", "pending_ownership_target_count", "sensitivity_target_count"]].sum(axis=1) == 1).all(), f"{len(denominator_targets)} target-period members reconciled", True, "Every target-period member must have exactly one analytical scope")
add_check("S3A014", "denominator_missingness", "Not-observed and explicit-unavailability target counts remain separate", not ((denominator_targets["not_observed_target_count"].eq(1)) & (denominator_targets["explicit_not_available_target_count"].eq(1))).any(), "No target is both not-observed and explicitly unavailable", True, "Keep absence of a row separate from an explicit unavailable row")
add_check("S3A015", "metric_semantics", "TBI, rank, bounds, booleans, and qualitative facts remain source-native and unpooled", not competitive_observation_universe["metric"].str.contains("market_share", case=False, regex=False).any() and competitive_observation_universe.loc[competitive_observation_universe["unit"].eq("rank"), "category_strength_candidate"].eq("no").all(), "No market-share relabeling or rank-as-interval eligibility", True, "Preserve source-native metric semantics")
add_check("S3A016", "consumer_reach", "Current limited Brand Footprint facts are not marked eligible for cross-group consumer-reach calculation", competitive_observation_universe["consumer_reach_group_comparison_eligible"].eq("no").all(), "0 observations eligible for consumer-reach group comparison", True, "Acquire a comparable CRP universe before consumer-reach comparison", "Limited public facts remain context or ordinal evidence.", True)
add_check("S3A017", "series_boundary", "Every longitudinal series is confined to one source label and one methodology cluster", longitudinal_series_eligibility["source_label_series_key"].is_unique and not longitudinal_series_eligibility["observed_periods"].str.contains("2025;2026", regex=False).any(), f"{len(longitudinal_series_eligibility)} bounded series", True, "Do not bridge source-label or methodology boundaries")
add_check("S3A018", "longitudinal_threshold", "Consistency and momentum eligibility requires at least three consecutive observed periods", longitudinal_series_eligibility.loc[longitudinal_series_eligibility["consistency_eligibility"].eq("eligible_with_caveat"), "maximum_consecutive_observed_periods"].astype(int).ge(3).all() and longitudinal_series_eligibility.loc[longitudinal_series_eligibility["momentum_eligibility"].eq("eligible_with_caveat"), "maximum_consecutive_observed_periods"].astype(int).ge(3).all(), f"{longitudinal_series_eligibility['consistency_eligibility'].eq('eligible_with_caveat').sum()} eligible consistency series", True, "Require three consecutive periods before calculation")
add_check("S3A019", "alias_protection", "Bango alias-review series remain source-label segments", longitudinal_series_eligibility.loc[longitudinal_series_eligibility["universe_id"].eq("AU018"), "identity_continuity_treatment"].eq("source_label_segment_only").all(), "Cap Bango and BANGO remain separate source-label segments", True, "Do not silently merge label segments")
add_check("S3A020", "persistence_baseline", "Persistence baselines are identified only from complete multi-group periods", not persistence_baseline_candidates["baseline_status"].eq("ambiguous_tie").any() and persistence_baseline_candidates.loc[persistence_baseline_candidates["baseline_status"].eq("identified"), "baseline_period"].ne("").all(), f"{persistence_baseline_candidates['baseline_status'].eq('identified').sum()} baselines identified", True, "Do not define baselines from incomplete or tied periods")
add_check("S3A021", "concentration", "Performance concentration remains ineligible under the current non-additive metric universe", stage3_metric_eligibility_rules.loc[stage3_metric_eligibility_rules["dimension"].eq("portfolio_concentration"), "current_eligibility"].eq("not_eligible_currently").all(), "HHI not eligible", True, "Do not calculate HHI from cross-category TBI or mixed metrics")
add_check("S3A022", "overall_leadership", "Overall portfolio leadership remains deferred", stage3_metric_eligibility_rules.loc[stage3_metric_eligibility_rules["dimension"].eq("overall_portfolio_leadership"), "current_eligibility"].eq("deferred").all(), "Overall winner deferred", True, "Complete validated dimension-level analysis before synthesis")
add_check("S3A023", "selective_coverage", "Selective public-source coverage remains an explicit structural limitation", True, "PASS_WITH_CAVEAT", False, "Use explicit eligible and observable denominators", "Coverage is a validation property, not a performance result.", True)
add_check("S3A024", "output_scope", "Stage 3A outputs contain preparation and eligibility fields rather than winner or composite-score fields", not any("winner" in column.casefold() or "composite_score" in column.casefold() for table in [structural_portfolio_universe, competitive_observation_universe, group_category_period_denominators, longitudinal_series_eligibility, persistence_baseline_candidates] for column in table.columns), "No winner or composite-score output fields", True, "Keep Stage 3A limited to preparation and eligibility")

stage3a_validation = pd.DataFrame(validation_rows)
critical_failures = stage3a_validation["critical_failure"].eq("yes").sum()
if critical_failures:
    print(stage3a_validation[["check_id", "validation_area", "result", "status"]].to_string(index=False))
    raise AssertionError(f"Stage 3A has {critical_failures} critical validation failure(s).")

caveat_count = stage3a_validation["status"].eq("passed_with_caveat").sum()
final_gate = "PASS_WITH_CAVEAT" if caveat_count else "PASS"
add_check(
    "S3A025",
    "stage_gate",
    "Stage 3A analytical preparation is complete enough for execution review",
    True,
    final_gate,
    False,
    "Review the executed notebook and generated analytical outputs before repository publication",
    "No portfolio winner, composite score, or final performance result is produced.",
    force_caveat=bool(caveat_count),
)
stage3a_validation = pd.DataFrame(validation_rows)

print(stage3a_validation[["check_id", "validation_area", "result", "status"]].to_string(index=False))
print(f"Final gate: {final_gate}")


check_id            validation_area                                                      result             status
  S3A001            input_integrity                                                  8/8 passed             passed
  S3A002           prior_stage_gate                             0 prior-stage critical failures             passed
  S3A003     ownership_preservation                                    157/157 records retained             passed
  S3A004       strict_control_scope                                         133 primary records             passed
  S3A005    nonprimary_preservation                                      24 non-primary records             passed
  S3A006       acquisition_universe                                         28 universe members             passed
  S3A007   observation_preservation                                   100 observations retained             passed
  S3A008     observation_uniqueness                                             

## Output Writing and Manifest

Write the approved analytical views, eligibility-rule registry, validation results, and a reproducible output manifest.


In [12]:
structural_path = ANALYTICAL_ROOT / "structural_portfolio_universe.csv"
competitive_path = ANALYTICAL_ROOT / "competitive_observation_universe.csv"
denominator_path = ANALYTICAL_ROOT / "group_category_period_denominators.csv"
longitudinal_path = ANALYTICAL_ROOT / "longitudinal_series_eligibility.csv"
persistence_path = ANALYTICAL_ROOT / "persistence_baseline_candidates.csv"
rules_path = METADATA_ROOT / "stage3_metric_eligibility_rules.csv"
validation_path = METADATA_ROOT / "stage3_preparation_validation.csv"
manifest_path = METADATA_ROOT / "stage3_output_manifest.csv"

structural_portfolio_universe.to_csv(structural_path, index=False)
competitive_observation_universe.to_csv(competitive_path, index=False)
group_category_period_denominators.to_csv(denominator_path, index=False)
longitudinal_series_eligibility.to_csv(longitudinal_path, index=False)
persistence_baseline_candidates.to_csv(persistence_path, index=False)
stage3_metric_eligibility_rules.to_csv(rules_path, index=False)
stage3a_validation.to_csv(validation_path, index=False)

manifest_rows = []
for path, description in [
    (structural_path, "Ownership-complete structural portfolio universe with strict-control and category-breadth eligibility flags"),
    (competitive_path, "Standardized competitive observations with primary, sensitivity, context, ownership-support, and method-candidate flags"),
    (denominator_path, "Group-category-period eligible, observable, observed, missing, context, and sensitivity denominators"),
    (longitudinal_path, "Source-label and methodology-bounded longitudinal series eligibility"),
    (persistence_path, "Objectively defined persistence baseline incumbent candidates without persistence outcomes"),
    (rules_path, "Pre-specified dimension-level method and denominator eligibility rules"),
    (validation_path, "Stage 3A analytical preparation validation results"),
]:
    manifest_rows.append(
        {
            "file_path": str(path.relative_to(OUTPUT_ROOT)),
            "row_count": str(
                len(pd.read_csv(path, dtype=str, keep_default_na=False))
            ),
            "sha256": sha256_file(path),
            "description": description,
            "repository_head": REPOSITORY_HEAD,
        }
    )

stage3_output_manifest = pd.DataFrame(manifest_rows)
stage3_output_manifest.to_csv(manifest_path, index=False)
print(stage3_output_manifest.to_string(index=False))


                                             file_path row_count                                                           sha256                                                                                                             description                          repository_head
     data/analytical/structural_portfolio_universe.csv       157 08efb815eb0fb7fafbff2b87ad3c2ec5c9481441ff3e15d481ce51315abb89ab             Ownership-complete structural portfolio universe with strict-control and category-breadth eligibility flags 5e64c377be2deea7c5b3568aefe2c12d9e1db2d8
  data/analytical/competitive_observation_universe.csv       100 9996d0d5c8db7587f1d6d84e2ba41118d1161ac9ca7dd98e317b80861fcdd1af Standardized competitive observations with primary, sensitivity, context, ownership-support, and method-candidate flags 5e64c377be2deea7c5b3568aefe2c12d9e1db2d8
data/analytical/group_category_period_denominators.csv        82 4add32e0731ad60f16c542fdb8d47e4ed9f239bb206dbff954a9b3fbbbe9f5